# 01_bronze — Load archived partner files into Bronze

Bronze keeps every partner row exactly as received: all columns as text, duplicates and bad values included.
Two columns are added: `_source_file` (file name) and `_ingested_at` (load time).

`COPY INTO` remembers which files it has already loaded. Running it again loads only new files.

## Step 1: Check the archived files
Lists every file Data Factory has moved to `archive/`, for all dates.

In [0]:
base = "abfss://settlements@insuranceadlsgenk2.dfs.core.windows.net/archive/2026/10/"
for day in dbutils.fs.ls(base):
    for f in dbutils.fs.ls(day.path):
        print(day.name, f.name, f.size)

## Step 2: Create the Bronze tables
Creates empty tables. Columns are added by `COPY INTO` the first time data is loaded.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS insurance.bronze.bank_settlement;
CREATE TABLE IF NOT EXISTS insurance.bronze.tpa_settlement;
CREATE TABLE IF NOT EXISTS insurance.bronze.repair_invoice;

## Step 3: Load BANK files

In [0]:
%sql
COPY INTO insurance.bronze.bank_settlement
FROM (
  SELECT *,
         _metadata.file_name   AS _source_file,
         current_timestamp()   AS _ingested_at
  FROM 'abfss://settlements@insuranceadlsgenk2.dfs.core.windows.net/archive/'
)
FILEFORMAT = CSV
PATTERN = '*/*/*/BANK_*.csv'
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true');

## Step 4: Load TPA files

In [0]:
%sql
COPY INTO insurance.bronze.tpa_settlement
FROM (
  SELECT *,
         _metadata.file_name   AS _source_file,
         current_timestamp()   AS _ingested_at
  FROM 'abfss://settlements@insuranceadlsgenk2.dfs.core.windows.net/archive/'
)
FILEFORMAT = CSV
PATTERN = '*/*/*/TPA_*.csv'
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true');

## Step 5: Load REPAIR files

In [0]:
%sql
COPY INTO insurance.bronze.repair_invoice
FROM (
  SELECT *,
         _metadata.file_name   AS _source_file,
         current_timestamp()   AS _ingested_at
  FROM 'abfss://settlements@insuranceadlsgenk2.dfs.core.windows.net/archive/'
)
FILEFORMAT = CSV
PATTERN = '*/*/*/REPAIR_*.csv'
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'true');

## Step 6: Check total row counts
Expected after Day 1 and Day 2: bank 40, tpa 30, repair 23.

In [0]:
%sql
SELECT 'bank' AS tbl, COUNT(*) AS cnt FROM insurance.bronze.bank_settlement
UNION ALL SELECT 'tpa', COUNT(*) FROM insurance.bronze.tpa_settlement
UNION ALL SELECT 'repair', COUNT(*) FROM insurance.bronze.repair_invoice;

## Step 7: Check row counts per file
Each file must match the row count in its manifest.

In [0]:
%sql
SELECT 'bank' AS tbl, _source_file, COUNT(*) AS cnt FROM insurance.bronze.bank_settlement GROUP BY _source_file
UNION ALL
SELECT 'tpa', _source_file, COUNT(*) FROM insurance.bronze.tpa_settlement GROUP BY _source_file
UNION ALL
SELECT 'repair', _source_file, COUNT(*) FROM insurance.bronze.repair_invoice GROUP BY _source_file
ORDER BY tbl, _source_file;

## Step 8: View the Day 2 bank rows
Shows the raw values that Silver will clean: currency symbols, mixed case, blank and `N/A` values, different date formats, and duplicates.

In [0]:
%sql
SELECT bank_ref, payment_id, claim_id, amount, settlement_date, status, _source_file
FROM insurance.bronze.bank_settlement
WHERE _source_file = 'BANK_20261012.csv';

In [0]:
display(dbutils.fs.ls("abfss://settlements@insuranceadlsgenk2.dfs.core.windows.net/archive/2026/10/11/"))